In [ ]:
"""
04_shap_analysis.py -- Scout AI Smart Inspector SHAP Explainer

Generates global feature importance explanations using SHAP values.
Automatically routes players to the appropriate model based on their
transfer history, ensuring the explanations perfectly match real-world
inference paths.
"""

import os
import sys
import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib
matplotlib.use('Agg')  # Save to file without opening interactive window
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
from dotenv import load_dotenv, find_dotenv

# ==========================================
# 0. CONFIG & SETUP
# ==========================================

# Load environment variables
load_dotenv(find_dotenv())

MODELS_DIR = "models"
IMAGES_DIR = "images"

# Ensure output directory exists
os.makedirs(IMAGES_DIR, exist_ok=True)

# ==========================================
# 1. CONSTANTS & FEATURE LISTS
# ==========================================

FULL_FEATURES = [
    'age', 'height_in_cm', 'total_appearances', 'minutes_per_match',
    'total_goals', 'total_assists', 'goals_per_90', 'assists_per_90',
    'total_yellow_cards', 'total_red_cards', 'international_caps', 'international_goals',
    'club_squad_size', 'club_avg_age', 'stadium_seats', 'foreigners_percentage',
    'contract_months_remaining', 'wonderkid_hype', 'league_coefficient',
    'has_transfer_history', 'max_career_transfer_fee', 'most_recent_transfer_fee',
    'detailed_position', 'foot', 'passport_tier',
]

PERFORMANCE_FEATURES = [
    'age', 'height_in_cm', 'total_appearances', 'minutes_per_match',
    'total_goals', 'total_assists', 'goals_per_90', 'assists_per_90',
    'total_yellow_cards', 'total_red_cards', 'international_caps', 'international_goals',
    'club_squad_size', 'club_avg_age', 'stadium_seats', 'foreigners_percentage',
    'wonderkid_hype', 'league_coefficient', 'detailed_position', 'foot', 'passport_tier',
]

NAME_MAP = {
    'remainder__age': 'Player Age',
    'remainder__max_career_transfer_fee': 'Max Transfer Fee',
    'remainder__most_recent_transfer_fee': 'Most Recent Transfer Fee',
    'remainder__international_caps': 'International Caps',
    'remainder__contract_months_remaining': 'Contract Remaining',
    'remainder__total_appearances': 'Total Appearances',
    'remainder__stadium_seats': 'Stadium Capacity',
    'remainder__minutes_per_match': 'Minutes Per Match',
    'remainder__foreigners_percentage': 'Foreigner Ratio',
    'remainder__has_transfer_history': 'Transfer History',
    'remainder__club_avg_age': 'Club Avg Age',
    'remainder__total_goals': 'Total Goals',
    'remainder__goals_per_90': 'Goals Per 90',
    'remainder__international_goals': 'International Goals',
    'remainder__club_squad_size': 'Squad Size',
    'remainder__total_assists': 'Total Assists',
    'remainder__height_in_cm': 'Height (cm)',
    'remainder__wonderkid_hype': 'Wonderkid Hype Index',
    'remainder__assists_per_90': 'Assists Per 90',
    'remainder__total_yellow_cards': 'Yellow Cards',
    'remainder__total_red_cards': 'Red Cards',
    'remainder__league_coefficient': 'League Quality',
    'cat__passport_tier_Tier_1': 'Passport (Tier 1)',
    'cat__detailed_position_Goalkeeper': 'Position: Goalkeeper',
}

def clean_feature_names(feature_names):
    return [
        NAME_MAP.get(name, name.replace('remainder__', '').replace('cat__', '').replace('_', ' ').title())
        for name in feature_names
    ]

# ==========================================
# 2. SHAP ANALYSIS FUNCTION
# ==========================================

def run_shap_analysis(model_label: str, feature_list: list, subset_df: pd.DataFrame):
    """Run and save a SHAP summary plot for one model."""
    print(f"[SYSTEM] [{model_label}] Loading model and computing SHAP values ({len(subset_df)} players)...")
    
    model_path = os.path.join(MODELS_DIR, f'scout_model_{model_label}.pkl')
    if not os.path.exists(model_path):
        print(f"[ERROR] Model not found at '{model_path}'. Please train models first.")
        return

    pipeline = joblib.load(model_path)
    preprocessor = pipeline.named_steps['preprocessor']
    regressor = pipeline.named_steps['regressor']

    X_transformed = preprocessor.transform(subset_df[feature_list])
    feature_names = preprocessor.get_feature_names_out()
    clean_names = clean_feature_names(feature_names)

    explainer = shap.TreeExplainer(regressor)
    shap_values = explainer(X_transformed)

    plt.figure(figsize=(10, 10))
    shap.summary_plot(shap_values, X_transformed, feature_names=clean_names, show=False)
    plt.title(f"ScoutAI ({model_label.replace('_', ' ').title()}): Global Feature Importance",
              fontsize=16, fontweight='bold', pad=20)
    plt.xlabel("SHAP Value (Impact on predicted market value)", fontsize=12)
    plt.tight_layout()
    
    out_path = os.path.join(IMAGES_DIR, f'shap_summary_{model_label}.png')
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"[SYSTEM] [{model_label}] Plot successfully saved to '{out_path}'.")


# ==========================================
# 3. MAIN EXECUTION
# ==========================================

def main():
    db_url = os.getenv('DB_URL')
    if not db_url:
        print("[ERROR] DB_URL not found. Please ensure your .env file exists and is configured correctly.")
        sys.exit(1)

    print("[SYSTEM] Connecting to database...")
    try:
        engine = create_engine(db_url)
        df = pd.read_sql("SELECT * FROM view_scout_master", engine)
    except Exception as e:
        print(f"[ERROR] Database connection failed: {e}")
        sys.exit(1)

    print("[SYSTEM] Performing robust feature engineering...")
    cols_to_clean = [
        'age', 'total_appearances', 'international_caps', 'international_goals',
        'max_career_transfer_fee', 'most_recent_transfer_fee', 'height_in_cm',
        'minutes_per_match', 'total_goals', 'total_assists', 'goals_per_90',
        'assists_per_90', 'total_yellow_cards', 'total_red_cards', 'club_squad_size',
        'club_avg_age', 'stadium_seats', 'foreigners_percentage', 'contract_months_remaining',
    ]
    
    for col in cols_to_clean:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    df['wonderkid_hype'] = np.where(df['age'] <= 22, (df['total_appearances'] + (df['international_caps'] * 3)), 0)

    conditions = [
        (df['passport_power_rank'].fillna(999) <= 15),
        (df['passport_power_rank'].fillna(999) > 15) & (df['passport_power_rank'].fillna(999) <= 60),
    ]
    df['passport_tier'] = np.select(conditions, ['Tier_1', 'Tier_2'], default='Tier_3')

    league_weights = {
        'Premier League': 1.5, 'LaLiga': 1.4, 'Serie A': 1.3, 'Bundesliga': 1.3,
        'Ligue 1': 1.2, 'Eredivisie': 1.15, 'Liga Portugal': 1.15, 'Süper Lig': 1.05,
    }
    df['league_coefficient'] = df['competition_name'].map(league_weights).fillna(1.0)
    df['detailed_position'] = df['sub_position'].fillna(df['position_group']).astype(str)

    # Segment data exactly as it happens in production inference
    df_full = df[df['has_transfer_history'] == 1].copy()
    df_perf = df[df['has_transfer_history'] == 0].copy()

    # Run for both models
    print("\n--- Generating SHAP Analysis ---")
    run_shap_analysis('full', FULL_FEATURES, df_full)
    run_shap_analysis('performance_only', PERFORMANCE_FEATURES, df_perf)
    
    print("\n[SYSTEM] All SHAP analyses completed successfully.")

if __name__ == "__main__":
    main()

C:\Users\ali-k\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[SYSTEM] Connecting to database...
[SYSTEM] Performing robust feature engineering...

--- Generating SHAP Analysis ---
[SYSTEM] [full] Loading model and computing SHAP values (9334 players)...
[SYSTEM] [full] Plot successfully saved to 'images\shap_summary_full.png'.
[SYSTEM] [performance_only] Loading model and computing SHAP values (11046 players)...
[SYSTEM] [performance_only] Plot successfully saved to 'images\shap_summary_performance_only.png'.

[SYSTEM] All SHAP analyses completed successfully.
